# OpenPlaque — BACCE teacher-constrained calibration

Self-contained notebook; no `%run`. This branch starts directly from `main`.

Goal: use the already reconstructed proximal RCA route as **ground truth only for calibration/evaluation**, not as a term in the tracker score. We search deterministic BACCE-style scoring weights that best reproduce the known route over the first ~15–20 mm after the 6-mm seed, then remove the teacher and test extension beyond the known route.

Pass target: median error < 1 mm and maximum error < 2 mm over the first 15 mm, while distance from the aorta does not collapse.


In [ ]:
# FIRST EXECUTABLE CELL
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!rm -rf /content/OpenPlaque /content/BACCE
!git clone -q --branch bacce-teacher-calibration-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git clone -q https://github.com/514sz/Branch-aware-centerline-extraction.git /content/BACCE
%pip -q install pydicom SimpleITK scipy scikit-image matplotlib pandas


In [ ]:
import sys, shutil, math, itertools
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, SimpleITK as sitk
from scipy import ndimage as ndi
from skimage.filters import frangi
from skimage.graph import route_through_array

sys.path.insert(0,'/content/OpenPlaque/src')
sys.path.insert(0,'/content/BACCE')
from openplaque.study import OpenPlaqueStudy
from utils import create_actions

ROOT=Path('/content/drive/MyDrive/OpenPlaque')
OUT=ROOT/'BACCE_Teacher_Calibration'
OUT.mkdir(parents=True,exist_ok=True)


## 1. Load source CCTA series 7 and TotalSegmentator aorta mask


In [ ]:
dz=ROOT/'Full_DICOM.zip'; lz=Path('/content/Full_DICOM.zip')
if not dz.exists(): raise FileNotFoundError(dz)
if not lz.exists() or lz.stat().st_size!=dz.stat().st_size: shutil.copyfile(dz,lz)
shutil.rmtree('/content/full_dicom_bacce_teacher',ignore_errors=True)
study=OpenPlaqueStudy(str(lz),extract_root='/content/full_dicom_bacce_teacher')
img,ct,_=study.load_series(7); ct=np.asarray(ct)
sp_xyz=np.array(img.GetSpacing(),float); sp_zyx=sp_xyz[::-1]

ap=ROOT/'RCA_Ostium_TotalSegmentator'/'aorta_series7_totalseg.nii.gz'
if not ap.exists(): raise FileNotFoundError(ap)
aim=sitk.ReadImage(str(ap))
if aim.GetSize()!=img.GetSize() or not np.allclose(aim.GetSpacing(),img.GetSpacing()):
    aim=sitk.Resample(aim,img,sitk.Transform(),sitk.sitkNearestNeighbor,0,sitk.sitkUInt8)
aorta=sitk.GetArrayFromImage(aim)>0
dist_aorta=ndi.distance_transform_edt(~aorta,sampling=sp_zyx)
print('shape:',ct.shape,' spacing xyz:',sp_xyz)


## 2. Recover the same ostium candidate and known proximal image-supported route


In [ ]:
zlo,zhi=225,305
band=np.zeros_like(aorta,bool); band[zlo:zhi+1]=True
a_hu=ct[aorta & band]
blood_thr=float(np.clip(np.median(a_hu)*0.43,180,360))
shell=(dist_aorta>0.8)&(dist_aorta<=8.0)&band
cand=(ct>=blood_thr)&shell
cand=ndi.binary_closing(cand,structure=np.ones((3,3,3),bool),iterations=1)
lab,n=ndi.label(cand,structure=np.ones((3,3,3),bool))
rows=[]
for k in range(1,n+1):
    c=np.argwhere(lab==k)
    if len(c)<5: continue
    dv=dist_aorta[tuple(c.T)]
    if dv.min()>3 or dv.max()<2: continue
    mm=c.astype(float)*sp_zyx
    if len(mm)<3: continue
    ev=np.sort(np.linalg.eigvalsh(np.cov(mm,rowvar=False)))[::-1]
    L=float(np.sqrt(max(1e-6,12*ev[0])))
    z0,y0,x0=c.min(0); z1,y1,x1=c.max(0)+1
    sub=(lab[z0:z1,y0:y1,x0:x1]==k)
    rmax=float(ndi.distance_transform_edt(sub,sampling=sp_zyx).max())
    if not (2<=L<=20 and rmax<=4.5): continue
    q=c[np.argmin(dv)]
    rows.append((k,L,rmax,float(dv.min()),float(dv.max()),q))
if not rows: raise RuntimeError('No ostium-like candidate')
rows=sorted(rows,key=lambda t:(t[1]+2*t[4]-2*t[2]),reverse=True)
k,L,rmax,mind,maxd,ostium=rows[0]
ostium=np.array(ostium,float)

pad=np.array([24,80,80]); lo=np.maximum(np.floor(ostium-pad).astype(int),0); hi=np.minimum(np.ceil(ostium+pad).astype(int)+1,np.array(ct.shape))
roi=ct[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]].astype(float)
ar=aorta[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]]
sm=ndi.gaussian_filter(roi,0.7)
v=frangi(sm,sigmas=[0.7,1.0,1.4,1.8],black_ridges=False)
v99=np.percentile(v[v>0],99) if np.any(v>0) else 1.0
vn=np.clip(v/max(v99,1e-6),0,1)
med_hu=float(np.median(ct[aorta & band]))
lohu=max(140.,blood_thr*.55); hihu=max(lohu+100.,med_hu*1.35)
ints=np.clip((sm-lohu)/(hihu-lohu),0,1)
support=.58*ints+.42*vn
da=ndi.distance_transform_edt(~ar,sampling=sp_zyx)
cost=1/(.06+support); cost[ar]+=40; cost[da<.6]+=15; cost[sm<lohu]+=12

start=np.round(ostium-lo).astype(int)
pts=np.argwhere((support>.40)&(da>6)&(da<26))
eps=[]
stride=max(1,len(pts)//2500)
for p in pts[::stride]:
    d=np.linalg.norm((p-start)*sp_zyx)
    if 16<=d<=32: eps.append((float(support[tuple(p)]),*map(int,p)))
eps=sorted(eps,reverse=True)[:150]
routes=[]
for s,ez,ey,ex in eps:
    try: path,_=route_through_array(cost,tuple(start),(ez,ey,ex),fully_connected=True,geometric=True)
    except Exception: continue
    p=np.asarray(path,int)
    step=np.diff(p.astype(float),axis=0)*sp_zyx; Lp=float(np.linalg.norm(step,axis=1).sum())
    if Lp<18: continue
    dd=da[tuple(p.T)]; su=support[tuple(p.T)]
    outward=float(np.mean(np.diff(dd)>=-.35)) if len(dd)>1 else 0
    score=.45*np.mean(su)+.25*outward+.20*np.tanh(dd[-1]/12)+.10*np.tanh(Lp/20)
    routes.append((score,Lp,p))
if not routes: raise RuntimeError('Could not reconstruct proximal route')
routes.sort(key=lambda t:t[0],reverse=True)
route=routes[0][2]; route_global=route+lo
seg=np.linalg.norm(np.diff(route_global.astype(float),axis=0)*sp_zyx,axis=1)
arc=np.r_[0,np.cumsum(seg)]
seed_idx=int(np.argmin(np.abs(arc-6.0)))
seed=route_global[seed_idx].astype(float)
print('known route length mm:',round(float(arc[-1]),2),' seed arc:',round(float(arc[seed_idx]),2),' seed:',seed)


## 3. Precompute full-volume image support maps


In [ ]:
ctf=ct.astype(np.float32)
sm_full=ndi.gaussian_filter(ctf,0.7)
mins=np.maximum(np.floor(route_global.min(0)-np.array([30,90,90])).astype(int),0)
maxs=np.minimum(np.ceil(route_global.max(0)+np.array([30,90,90])).astype(int)+1,np.array(ct.shape))
sl=tuple(slice(a,b) for a,b in zip(mins,maxs))
crop=sm_full[sl]
vf=frangi(crop,sigmas=[0.7,1.0,1.4,1.8],black_ridges=False)
vv=np.zeros_like(ctf,dtype=np.float32)
vv[sl]=vf.astype(np.float32)
vp=np.percentile(vf[vf>0],99) if np.any(vf>0) else 1.0
vv=np.clip(vv/max(vp,1e-6),0,1)
bright=np.clip((sm_full-lohu)/(hihu-lohu),0,1).astype(np.float32)
tube=ndi.gaussian_filter(bright,1.0).astype(np.float32)
print('support maps ready')


## 4. BACCE 500-action lattice and tracker


In [ ]:
actions=create_actions(500,dis=1.5,spacing=sp_xyz).astype(float)
actions=np.unique(actions,axis=0)
actions=actions[np.linalg.norm(actions*sp_zyx,axis=1)>0]
print('unique nonzero actions:',len(actions))

def interp(arr,p):
    return float(ndi.map_coordinates(arr,np.asarray(p,float)[:,None],order=1,mode='nearest')[0])

def angle_cos(a,b):
    aa=a*sp_zyx; bb=b*sp_zyx
    na=np.linalg.norm(aa); nb=np.linalg.norm(bb)
    if na<1e-8 or nb<1e-8: return -1.0
    return float(np.clip(np.dot(aa,bb)/(na*nb),-1,1))

def nearest_known_mm(p, known):
    d=(known-p)*sp_zyx
    return float(np.min(np.linalg.norm(d,axis=1)))

def tracker_run(weights, max_mm=18.0, extend=False):
    wB,wV,wT,wF,wO,wC=weights
    i0=int(np.argmin(np.abs(arc-3.0))); i1=int(np.argmin(np.abs(arc-9.0)))
    prev=(route_global[i1]-route_global[i0]).astype(float)
    cur=seed.copy(); pts=[cur.copy()]; logs=[]
    traveled=0.0
    max_total=60.0 if extend else max_mm
    for stepno in range(120):
        scored=[]
        cur_da=interp(dist_aorta,cur)
        for a in actions:
            p=cur+a
            if np.any(p<2) or np.any(p>=np.array(ct.shape)-2): continue
            c=angle_cos(a,prev)
            if c < math.cos(math.radians(75)): continue
            b=interp(bright,p); ve=interp(vv,p); tu=interp(tube,p)
            d=interp(dist_aorta,p)
            outward=np.tanh((d-cur_da)/1.5)
            ss=[]
            for u in np.linspace(.2,1.0,5):
                q=cur+u*a
                ss.append(.55*interp(bright,q)+.25*interp(vv,q)+.20*interp(tube,q))
            cont=float(np.mean(ss))
            score=wB*b+wV*ve+wT*tu+wF*max(0,c)+wO*outward+wC*cont
            if b<.10 and tu<.15: score-=1.5
            scored.append((score,p,a,b,ve,tu,d,c,cont))
        if not scored: break
        scored.sort(key=lambda x:x[0],reverse=True)
        best=scored[0]
        _,nxt,a,b,ve,tu,d,c,cont=best
        stepmm=float(np.linalg.norm((nxt-cur)*sp_zyx))
        if stepmm<.2: break
        traveled+=stepmm
        logs.append(best)
        pts.append(nxt.copy()); cur=nxt; prev=a
        if traveled>=max_total: break
        if best[0] < 0.20 and traveled>3: break
    return np.asarray(pts), logs

def agreement_metrics(path):
    p=path
    step=np.linalg.norm(np.diff(p,axis=0)*sp_zyx,axis=1) if len(p)>1 else np.array([])
    parc=np.r_[0,np.cumsum(step)] if len(p) else np.array([0.])
    mask=parc<=15.0
    errs=np.array([nearest_known_mm(q,route_global[seed_idx:]) for q in p[mask]])
    if len(errs)==0: return dict(median=np.inf,max=np.inf,n=0,length=0)
    return dict(median=float(np.median(errs)),max=float(np.max(errs)),n=len(errs),length=float(parc[mask][-1]))


## 5. Teacher-guided calibration of image-score weights


In [ ]:
# The teacher is used only to rank weight sets after running the tracker.
# It is NOT included in per-step score.
weight_sets=[]
for wB in [0.20,0.30,0.40]:
  for wV in [0.10,0.20,0.30]:
    for wT in [0.10,0.20]:
      for wF in [0.15,0.25,0.35]:
        for wO in [0.05,0.10]:
          wC=max(0.05,1.0-(wB+wV+wT+wF+wO))
          w=np.array([wB,wV,wT,wF,wO,wC],float)
          w=w/w.sum()
          weight_sets.append(tuple(w))

cal=[]
for i,w in enumerate(weight_sets):
    p,lg=tracker_run(w,max_mm=18.0,extend=False)
    m=agreement_metrics(p)
    plen=float(np.linalg.norm(np.diff(p,axis=0)*sp_zyx,axis=1).sum()) if len(p)>1 else 0.
    final_da=interp(dist_aorta,p[-1]) if len(p) else np.nan
    obj=m['median'] + .35*m['max'] + max(0,15-m['length'])*.25 + max(0,3-final_da)*.15
    cal.append({'objective':obj,'median_err_mm':m['median'],'max_err_mm':m['max'],
                'eval_length_mm':m['length'],'path_length_mm':plen,'final_dist_aorta_mm':final_da,
                'w_bright':w[0],'w_vessel':w[1],'w_tube':w[2],'w_forward':w[3],'w_outward':w[4],'w_continuity':w[5]})
caldf=pd.DataFrame(cal).sort_values('objective').reset_index(drop=True)
display(caldf.head(12))
caldf.to_csv(OUT/'calibration_grid.csv',index=False)
bestrow=caldf.iloc[0]
bestw=tuple(bestrow[['w_bright','w_vessel','w_tube','w_forward','w_outward','w_continuity']].to_numpy(float))
print('BEST WEIGHTS:',bestw)


## 6. Validate the best calibrated tracker, then extend without teacher


In [ ]:
known_path, known_logs=tracker_run(bestw,max_mm=20.0,extend=False)
km=agreement_metrics(known_path)
PASS = (km['median']<1.0 and km['max']<2.0 and km['length']>=12.0)
print('calibration validation:',km,' PASS=',PASS)

ext_path,ext_logs=tracker_run(bestw,max_mm=20.0,extend=True)
ext_step=np.linalg.norm(np.diff(ext_path,axis=0)*sp_zyx,axis=1) if len(ext_path)>1 else np.array([])
ext_arc=np.r_[0,np.cumsum(ext_step)]
ext_da=np.array([interp(dist_aorta,p) for p in ext_path])
print('free extension length from 6-mm seed:',round(float(ext_arc[-1]),2),'mm')
print('final distance to aorta:',round(float(ext_da[-1]),2),'mm')


## 7. QC figures and summary


In [ ]:
z=int(round(seed[0]))
fig,axs=plt.subplots(1,4,figsize=(20,5))

axs[0].imshow(ct[z],cmap='gray',vmin=-200,vmax=900)
axs[0].plot(route_global[:,2],route_global[:,1],'.',ms=2,label='known proximal')
axs[0].plot(known_path[:,2],known_path[:,1],'-',lw=2,label='calibrated')
axs[0].plot(ext_path[:,2],ext_path[:,1],'-',lw=1.5,label='free extension')
axs[0].plot(seed[2],seed[1],'o',ms=7,label='seed')
axs[0].set_title(f'axial z={z}'); axs[0].axis('off'); axs[0].legend(fontsize=7)

kstep=np.linalg.norm(np.diff(known_path,axis=0)*sp_zyx,axis=1) if len(known_path)>1 else np.array([])
karc=np.r_[0,np.cumsum(kstep)]
errs=np.array([nearest_known_mm(q,route_global[seed_idx:]) for q in known_path])
axs[1].plot(karc,errs); axs[1].axhline(1,ls='--'); axs[1].axhline(2,ls='--')
axs[1].set_xlabel('tracker arc length (mm)'); axs[1].set_ylabel('distance to known route (mm)'); axs[1].set_title('agreement QC')

axs[2].plot(ext_arc,ext_da)
axs[2].axvline(max(0,float(arc[-1]-arc[seed_idx])),ls='--',label='end known route')
axs[2].set_xlabel('arc length from 6-mm seed (mm)'); axs[2].set_ylabel('distance to aorta (mm)'); axs[2].set_title('free extension QC'); axs[2].legend(fontsize=7)

top=caldf.head(12)
axs[3].plot(np.arange(len(top)),top.median_err_mm,'o-',label='median')
axs[3].plot(np.arange(len(top)),top.max_err_mm,'o-',label='max')
axs[3].axhline(1,ls='--'); axs[3].axhline(2,ls='--')
axs[3].set_xlabel('top calibration rank'); axs[3].set_ylabel('error (mm)'); axs[3].set_title('weight search'); axs[3].legend(fontsize=7)

plt.tight_layout()
out=OUT/'bacce_teacher_calibration_report.png'
fig.savefig(out,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)

summary=pd.DataFrame([{
    'pass_target':PASS,
    'known_route_total_mm':float(arc[-1]),
    'seed_arc_from_ostium_mm':float(arc[seed_idx]),
    'median_error_first15_mm':km['median'],
    'max_error_first15_mm':km['max'],
    'evaluated_length_mm':km['length'],
    'free_extension_from_seed_mm':float(ext_arc[-1]),
    'free_extension_total_from_ostium_mm':float(arc[seed_idx]+ext_arc[-1]),
    'final_dist_aorta_mm':float(ext_da[-1]),
    **{k:float(bestrow[k]) for k in ['w_bright','w_vessel','w_tube','w_forward','w_outward','w_continuity']}
}])
display(summary)
summary.to_csv(OUT/'bacce_teacher_calibration_summary.csv',index=False)
print('Saved:',out)
